# Project 5 — MOJ Criminal Court Statistics · **Bronze layer**

**What this notebook does, in one breath:** the government gave us four Excel files.
On screen they look like small summary tables, but the *real* raw data is hidden
inside each file in a locked filing cabinet Excel calls the **pivot cache**. This
notebook opens each file, copies every raw row out of that cabinet, and saves it as
a fast, clean **Parquet** file — *without changing a single number*. That faithful
copy is the **Bronze** layer.

**Rules for Bronze:** no renaming, no fixing labels, no maths. Just an honest
photocopy, stamped with where it came from. (All the cleaning happens later, in Silver.)

Run the cells top to bottom. Each one prints something so you can *see* what happened.

## Step 0 — Tools I need

`pandas` (tables), `pyarrow` (writes Parquet), plus a few built-ins. If the import
fails, uncomment the pip line, run it once, then re-run this cell.

In [1]:
# If anything is missing, uncomment the next line, run once, then re-run this cell:
# %pip install pandas pyarrow

import zipfile                      # an .xlsx is secretly a ZIP of XML files
import hashlib                      # to fingerprint each source file (proves it's untampered)
import datetime as dt              # to timestamp when I made the copy
import xml.etree.ElementTree as ET  # to read the XML hidden inside the xlsx
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# The XML inside an xlsx tags everything with this long prefix. I save it to NS
# so the rest of my code stays short and readable.
NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
print("Tools loaded OK  ·  pandas", pd.__version__, "· pyarrow", pa.__version__)

Tools loaded OK  ·  pandas 2.2.2 · pyarrow 14.0.2


## Step 1 — Point at my folders

- **PROJECT_DIR** — the top of the project (the `moj-crown-court` folder).
- **BRONZE_DIR** — where the four raw Excel files live (my *in* tray).
- **OUT_DIR** — where the clean Parquet copies will go (my *out* tray).
- **RELEASE** — which quarter this data is from. I stamp every row with `"2025Q4"`
  so that when MOJ publishes next quarter I can tell the two apart.

Jupyter usually treats the notebook's own folder as the "current folder", so I step
up one level from `notebooks/` to reach the project root. The cell checks the files
are really there and tells me if they're not.

In [3]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":          # if I launched from the notebooks/ folder
    PROJECT_DIR = PROJECT_DIR.parent         # step up to the project root

BRONZE_DIR = PROJECT_DIR / "data" / "bronze"
OUT_DIR    = BRONZE_DIR / "parquet"
RELEASE    = "2025Q4"
OUT_DIR.mkdir(parents=True, exist_ok=True)   # make the out tray if it doesn't exist

FILES = [
    "cc_rdos_tool.xlsx",
    "cc_open_tool.xlsx",
    "cc_waiting_hearing_tool.xlsx",
    "timeliness_tool_Crown_Court.xlsx",
]

print("Project :", PROJECT_DIR)
print("Raw in  :", BRONZE_DIR)
print("Out     :", OUT_DIR, "\n")
for f in FILES:
    p = BRONZE_DIR / f
    print(("  found " if p.exists() else "  MISSING "), f, f"({p.stat().st_size/1e6:.1f} MB)" if p.exists() else "")
# If you see MISSING, put the four .xlsx files into data/bronze/ and re-run.

Project : /Users/yusufismail/moj-crown-court
Raw in  : /Users/yusufismail/moj-crown-court/data/bronze
Out     : /Users/yusufismail/moj-crown-court/data/bronze/parquet 

  found  cc_rdos_tool.xlsx (2.1 MB)
  found  cc_open_tool.xlsx (23.8 MB)
  found  cc_waiting_hearing_tool.xlsx (38.7 MB)
  found  timeliness_tool_Crown_Court.xlsx (34.7 MB)


## Step 2 — Look inside ONE file's hidden cabinet

Let's start small with `cc_rdos_tool.xlsx` (the backlog-volumes file, only ~2 MB).
An `.xlsx` is really a zip, so I can list what's inside. The bits I care about are
the **pivotCache** files:
- `pivotCacheDefinition1.xml` — the **recipe card** (column names + value lists)
- `pivotCacheRecords1.xml` — the **actual rows**

In [6]:
demo = zipfile.ZipFile(BRONZE_DIR / "cc_rdos_tool.xlsx")
cache_bits = [n for n in demo.namelist() if "pivotCache" in n and n.endswith(".xml")]
print("Pivot-cache parts inside cc_rdos_tool.xlsx:")
for n in cache_bits:
    print("  ", n)

Pivot-cache parts inside cc_rdos_tool.xlsx:
   xl/pivotCache/pivotCacheDefinition1.xml
   xl/pivotCache/pivotCacheRecords1.xml


## Step 3 — Read the recipe card (`read_definition`)

The recipe card tells me two things: the **column names**, and — for text columns —
the **full list of allowed values**. That last part matters because text is stored
in a sneaky way: instead of writing `"Birmingham"` a thousand times, Excel writes a
number that *points at* the list (e.g. "court #12"). So I keep the list to translate
those numbers back into words later.

I define a function `read_definition(...)` that returns:
- `field_names` — the column names
- `shared` — for each column, its value list (empty `[]` if it's a number column)
- `is_text` — True if the column translates via the list
- `record_count` — how many rows the file *says* it has (I'll check against this)

In [8]:
def read_definition(zf, cache_no):
    root = ET.fromstring(zf.read(f"xl/pivotCache/pivotCacheDefinition{cache_no}.xml"))
    field_names, shared, is_text = [], [], []
    for cf in root.iter(f"{NS}cacheField"):
        field_names.append(cf.get("name"))
        si = cf.find(f"{NS}sharedItems")
        items = [e.get("v") for e in si] if si is not None else []
        shared.append(items)
        is_text.append(len(items) > 0)   # has a value list => it's a text column
    record_count = int(root.get("recordCount")) if root.get("recordCount") else None
    return field_names, shared, is_text, record_count

fields, shared, is_text, rc = read_definition(demo, 1)
print("Columns in cc_rdos:", fields)
print("Row count the file claims:", f"{rc:,}")
print("\nExample value list (crown_court) — first 8 of", len(shared[fields.index('crown_court')]), ":")
print(shared[fields.index('crown_court')][:8])

Columns in cc_rdos: ['year', 'quarter', 'receipt_type', 'region', 'lcjb_area', 'crown_court', 'rdos', 'offence_group', 'value']
Row count the file claims: 345,590

Example value list (crown_court) — first 8 of 71 :
['Central Criminal Court', 'Southwark', 'Snaresbrook', 'Harrow', 'Wood Green', 'Croydon', 'Inner London Sessions House', 'Woolwich']


## Step 4 — Teach the code to translate ONE cell (`cell_to_value`)

Every cell in a row is one of a few shapes:
- `<x v="12"/>` → a **text** cell: look up item #12 in the recipe card
- `<n v="8"/>`  → a **number** (an actual count) → keep it
- `<s v="..."/>`→ inline text → keep it
- `<m/>`        → **blank/missing** → `None`

`cell_to_value(...)` handles all of these.

In [10]:
def cell_to_value(cell, lookup):
    tag = cell.tag.rsplit('}', 1)[1]
    if tag == 'x':                       # points at the value list
        return lookup[int(cell.get('v'))]
    if tag == 'n':                       # a real number (a count)
        v = cell.get('v'); return float(v) if v not in (None, '') else None
    if tag == 's':                       # inline text
        return cell.get('v')
    if tag == 'b':                       # true / false
        return cell.get('v') in ('1', 'true', 'True')
    if tag in ('m', 'e'):                # missing or error => blank
        return None
    return cell.get('v')
print("cell_to_value() is ready.")

cell_to_value() is ready.


## Step 5 — Peek at the first 5 real rows (before copying millions)

Good habit: *look before you leap.* I stream just the first 5 rows of `cc_rdos`,
translate them, and eyeball them as a small table. This is where I confirm the data
matches what our structure notes said (year, quarter, region, court, rdos, offence,
and a `value` count).

In [12]:
def first_rows(xlsx_name, cache_no, n=5):
    zf = zipfile.ZipFile(BRONZE_DIR / xlsx_name)
    fields, shared, is_text, _ = read_definition(zf, cache_no)
    rows = []
    with zf.open(f"xl/pivotCache/pivotCacheRecords{cache_no}.xml") as fh:
        for _e, el in ET.iterparse(fh, events=("end",)):
            if not el.tag.endswith('}r'):
                continue
            cells = list(el)
            rows.append([cell_to_value(cells[i], shared[i]) for i in range(len(fields))])
            el.clear()
            if len(rows) >= n:
                break
    zf.close()
    return pd.DataFrame(rows, columns=fields)

first_rows("cc_rdos_tool.xlsx", 1, 5)

,year,quarter,receipt_type,region,lcjb_area,crown_court,rdos,offence_group,value
0,2016,Q1,01. Triable-either-way trials: total,London,Central London LCJB,Central Criminal Court,1. Receipts,00: All offences,63.0
1,2016,Q1,01. Triable-either-way trials: total,London,Central London LCJB,Central Criminal Court,1. Receipts,01: Violence against the person,4.0
2,2016,Q1,01. Triable-either-way trials: total,London,Central London LCJB,Central Criminal Court,1. Receipts,02: Sexual offences,1.0
3,2016,Q1,01. Triable-either-way trials: total,London,Central London LCJB,Central Criminal Court,1. Receipts,04: Theft offences,4.0
4,2016,Q1,01. Triable-either-way trials: total,London,Central London LCJB,Central Criminal Court,1. Receipts,06: Drug offences,21.0


## Step 6 — The full photocopy machine (`ingest_cache`)

Now the real thing. A few supporting pieces first:

- `FRIENDLY` — a plain-name for each cabinet. Most files have one; `cc_open_tool`
  has **two** (one for age-band detail, one for average durations), so I name them
  clearly instead of "cache1/cache2".
- `sha256_of(...)` — the file's **fingerprint** (provenance).
- `build_schema(...)` — the Parquet "shape": text columns as text, number columns as
  decimals, plus my five provenance columns (all start with `_`).
- `ingest_cache(...)` — streams every row, stamps provenance, and writes to Parquet
  **in batches** so memory stays flat (this is what avoids the out-of-memory crash).

Notice the **NA safety net** inside: MOJ sometimes writes a missing value as the word
`"NA"` inside a number column. A number column must hold numbers, so I turn anything
that isn't a real number into a blank. (That exact bug crashed our first run — now it's handled.)

In [14]:
FRIENDLY = {
    ("cc_rdos_tool.xlsx", 1):                "cc_rdos",
    ("cc_open_tool.xlsx", 1):                "cc_open_age",
    ("cc_open_tool.xlsx", 2):                "cc_open_averages",
    ("cc_waiting_hearing_tool.xlsx", 1):     "cc_waiting_hearing",
    ("timeliness_tool_Crown_Court.xlsx", 1): "cc_timeliness",
}
BATCH_ROWS = 250_000   # how many rows I hold before flushing to disk

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def build_schema(field_names, is_text):
    cols = [(n, pa.string() if t else pa.float64()) for n, t in zip(field_names, is_text)]
    cols += [("_source_file", pa.string()), ("_release", pa.string()),
             ("_cache", pa.string()), ("_source_sha256", pa.string()),
             ("_ingested_at", pa.string())]
    return pa.schema(cols)

def ingest_cache(xlsx_name, cache_no, file_hash, ingested_at):
    friendly  = FRIENDLY[(xlsx_name, cache_no)]
    zf        = zipfile.ZipFile(BRONZE_DIR / xlsx_name)
    fields, shared, is_text, rc = read_definition(zf, cache_no)
    schema    = build_schema(fields, is_text)
    out_path  = OUT_DIR / f"{friendly}.parquet"

    prov = {"_source_file": xlsx_name, "_release": RELEASE, "_cache": friendly,
            "_source_sha256": file_hash, "_ingested_at": ingested_at}

    writer, batch, written = pq.ParquetWriter(out_path, schema, compression="snappy"), [], 0
    def flush():
        nonlocal batch
        if not batch:
            return
        columns = {name: [] for name in schema.names}
        for row in batch:
            for name in schema.names:
                columns[name].append(row.get(name))
        writer.write_table(pa.table(columns, schema=schema))
        batch = []

    with zf.open(f"xl/pivotCache/pivotCacheRecords{cache_no}.xml") as fh:
        for _e, el in ET.iterparse(fh, events=("end",)):
            if not el.tag.endswith('}r'):
                continue
            cells = list(el)
            row = dict(prov)
            for i, name in enumerate(fields):
                v = cell_to_value(cells[i], shared[i])
                if not is_text[i]:                     # NA safety net for number columns
                    try:
                        v = float(v) if v not in (None, "") else None
                    except (TypeError, ValueError):
                        v = None
                row[name] = v
            batch.append(row); written += 1
            if len(batch) >= BATCH_ROWS:
                flush()
            el.clear()
    flush(); writer.close(); zf.close()
    return {"cache": friendly, "source_file": xlsx_name, "output": out_path.name,
            "rows_written": written, "rows_expected": rc, "columns": len(fields),
            "sha256": file_hash}

print("Photocopy machine ready.")

Photocopy machine ready.


## Step 7 — Run it on all four files

This streams ~9 million rows in total, so give it a minute or two. Each line prints
how many rows it wrote and whether that **matches** the count the file claimed — if
they don't match, something's wrong and I stop and look.

The biggest file (`cc_waiting_hearing`, ~4.3M rows) is the slow one.

In [16]:
ingested_at = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
manifest = []
for xlsx_name in FILES:
    file_hash = sha256_of(BRONZE_DIR / xlsx_name)
    for cache_no in sorted(c for (f, c) in FRIENDLY if f == xlsx_name):
        print(f"-> copying {xlsx_name} (cabinet {cache_no}) ...")
        info = ingest_cache(xlsx_name, cache_no, file_hash, ingested_at)
        ok = "OK" if info["rows_expected"] in (None, info["rows_written"]) else "MISMATCH!"
        print(f"   {info['cache']:<20} {info['rows_written']:>10,} rows  [{ok}] -> {info['output']}")
        manifest.append(info)

manifest = pd.DataFrame(manifest)
manifest.to_csv(OUT_DIR / "_bronze_manifest.csv", index=False)
print(f"\nDone. {len(manifest)} Parquet files, {manifest['rows_written'].sum():,} total rows.")
manifest

-> copying cc_rdos_tool.xlsx (cabinet 1) ...
   cc_rdos                 345,590 rows  [OK] -> cc_rdos.parquet
-> copying cc_open_tool.xlsx (cabinet 1) ...
   cc_open_age           3,129,789 rows  [OK] -> cc_open_age.parquet
-> copying cc_open_tool.xlsx (cabinet 2) ...
   cc_open_averages        395,534 rows  [OK] -> cc_open_averages.parquet
-> copying cc_waiting_hearing_tool.xlsx (cabinet 1) ...
   cc_waiting_hearing    4,346,919 rows  [OK] -> cc_waiting_hearing.parquet
-> copying timeliness_tool_Crown_Court.xlsx (cabinet 1) ...
   cc_timeliness           754,795 rows  [OK] -> cc_timeliness.parquet

Done. 5 Parquet files, 8,972,627 total rows.


,cache,source_file,output,rows_written,rows_expected,columns,sha256
0,cc_rdos,cc_rdos_tool.xlsx,cc_rdos.parquet,345590,345590,9,c35e348c85cb1047c732a0ec8d905530b267438150efba...
1,cc_open_age,cc_open_tool.xlsx,cc_open_age.parquet,3129789,3129789,7,d7d8cedea83c51652cb70b0af9a054e9cde22680e143b1...
2,cc_open_averages,cc_open_tool.xlsx,cc_open_averages.parquet,395534,395534,9,d7d8cedea83c51652cb70b0af9a054e9cde22680e143b1...
3,cc_waiting_hearing,cc_waiting_hearing_tool.xlsx,cc_waiting_hearing.parquet,4346919,4346919,11,f2729157396b3b359d92db624dfcd7b53713e21fdd2511...
4,cc_timeliness,timeliness_tool_Crown_Court.xlsx,cc_timeliness.parquet,754795,754795,30,879b81f5fb6da552f040df0f204518e86e02e8bef8604a...


## Step 8 — Prove the copy is faithful (rebuild the **80,203** headline)

This is the whole point of Bronze: the numbers must survive the copy untouched.
The government said the outstanding Crown Court caseload hit **~80,200** at the end
of December 2025. I rebuild that exact figure from the Parquet I just wrote. If it
comes back **80,203**, the photocopy is honest.

(Reminder from our notes: *Total cases* 80,203 vs *Valid cases* 75,799 — the age
bands only cover valid cases, and "1 year or more" = the `1 to 2 years` + `2 years or
more` bands.)

In [17]:
open_age = pd.read_parquet(OUT_DIR / "cc_open_age.parquet")

slice_ = open_age[(open_age.year == "2025") & (open_age.quarter == "Q4") &
                  (open_age.receipt_type == "01. All open cases") &
                  (open_age.offence_group == "00: All offences") &
                  (open_age.geographic_area == "England and Wales")]

total = slice_.loc[slice_.age_open_grouped == "Total cases", "value"].sum()
yr1   = slice_.loc[slice_.age_open_grouped.isin(["08: 1 to 2 years",
                                                 "09: 2 years or more"]), "value"].sum()

print(f"Total open cases (Q4 2025, E&W, all offences): {total:,.0f}   (published ~80,200)")
print(f"Open 1 year or more (valid basis)            : {yr1:,.0f}")
assert int(total) == 80203, "Mismatch — the copy is NOT faithful, stop and investigate!"
print("\n MATCH — Bronze is faithful. ")

Total open cases (Q4 2025, E&W, all offences): 80,203   (published ~80,200)
Open 1 year or more (valid basis)            : 21,002

 MATCH — Bronze is faithful. 


## Step 9 — Check the provenance stamp, then we're done

Every row carries where it came from. Let's confirm, then look at the manifest receipt.

In [20]:
cols = ["_source_file", "_release", "_cache", "_ingested_at", "_source_sha256"]
print("Provenance on the first row of cc_open_age:")
for k in cols:
    val = open_age[k].iloc[0]
    print(f"  {k:<16}: {val[:24] + '...' if k=='_source_sha256' else val}")

print("\nManifest (the receipt of everything Bronze produced):")
pd.read_csv(OUT_DIR / "_bronze_manifest.csv")

Provenance on the first row of cc_open_age:
  _source_file    : cc_open_tool.xlsx
  _release        : 2025Q4
  _cache          : cc_open_age
  _ingested_at    : 2026-08-07T09:11:33+00:00
  _source_sha256  : d7d8cedea83c51652cb70b0a...

Manifest (the receipt of everything Bronze produced):


,cache,source_file,output,rows_written,rows_expected,columns,sha256
0,cc_rdos,cc_rdos_tool.xlsx,cc_rdos.parquet,345590,345590,9,c35e348c85cb1047c732a0ec8d905530b267438150efba...
1,cc_open_age,cc_open_tool.xlsx,cc_open_age.parquet,3129789,3129789,7,d7d8cedea83c51652cb70b0af9a054e9cde22680e143b1...
2,cc_open_averages,cc_open_tool.xlsx,cc_open_averages.parquet,395534,395534,9,d7d8cedea83c51652cb70b0af9a054e9cde22680e143b1...
3,cc_waiting_hearing,cc_waiting_hearing_tool.xlsx,cc_waiting_hearing.parquet,4346919,4346919,11,f2729157396b3b359d92db624dfcd7b53713e21fdd2511...
4,cc_timeliness,timeliness_tool_Crown_Court.xlsx,cc_timeliness.parquet,754795,754795,30,879b81f5fb6da552f040df0f204518e86e02e8bef8604a...


---
### Bronze layer complete

The five source workbooks are now faithful Parquet extracts in `data/bronze/parquet/`, alongside a manifest recording each table's row count, its source-file fingerprint (SHA-256) and the time I ingested it.

I validated the extract by reconstructing the published Crown Court outstanding-caseload figure — **80,203** for the quarter ending December 2025 — directly from the Parquet. It matches the official release to the row, so I'm confident nothing was dropped or altered in the copy.

**Next — the Silver layer.** This is where I turn the raw extract into analysis-ready tables: friendly column names; compound labels such as `"04. Indictable only trials: remanded in custody"` split into separate `case_type` and `remand_status` fields; the open-caseload age bands ordered correctly; the `Annual` / `All` aggregate rows removed so nothing is double-counted; the offence-label spelling differences reconciled across files; and a proper `date` derived from year and quarter.